<a href="https://colab.research.google.com/github/KaevienAsoran/llm-from-foundations/blob/main/week02_rnn_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
sentences = [
    ["the", "cat", "drinks", "milk"],
    ["the", "dog", "drinks", "water"]
]

vocab = sorted(
    set(word for sentence in sentences for word in sentence)
)

word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}

print(vocab)
print(word_to_id)

In [ ]:
inputs = []
targets = []

for sentence in sentences:

    token_ids = [
        word_to_id[word]
        for word in sentence
    ]

    inputs.append(token_ids[:-1])
    targets.append(token_ids[1:])

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

print("Inputs:")
print(inputs)

print("\nTargets:")
print(targets)

print("\nInput shape:", inputs.shape)
print("Target shape:", targets.shape)

inputs.append(token_ids[:-1])：

含义：截取除最后一个词以外的所有词作为模型输入。

targets.append(token_ids[1:])：

含义：截取除第一个词以外的所有词作为预测目标。

inputs.shape  = [2, 3]

targets.shape = [2, 3]

2 = 两句话

3 = 每句话有三个 prediction positions

# RNN 模块正式开始

In [ ]:
class RNNLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_size,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, x):

        x = self.embedding(x)

        output, hidden = self.rnn(x)

        logits = self.output_layer(output)

        return logits, hidden

In [ ]:
model = RNNLanguageModel(
    vocab_size=len(vocab),
    embedding_dim=8,
    hidden_size=16
)

print(model)

Token IDs

   ↓

Embedding

   ↓

RNN

   ↓

Hidden States

   ↓

Linear Layer

   ↓

Logits

   ↓

预测下一个 token


# 先跑一次，不训练

In [ ]:
logits, hidden = model(inputs)

print("Logits shape:", logits.shape)
print("Hidden shape:", hidden.shape)

# 开始真正训练
先从optimizer开始

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)

In [ ]:
for epoch in range(300):

    optimizer.zero_grad()

    logits, hidden = model(inputs)

    loss = F.cross_entropy(
        logits.reshape(-1, len(vocab)),
        targets.reshape(-1)
    )

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.4f}"
        )

optimizer.zero_grad()

↓

清除旧梯度


model(inputs)

↓

forward


cross_entropy

↓
计算 next-token prediction 的误差


loss.backward()

↓

计算 gradient


optimizer.step()

↓

真正更新 Embedding + RNN + Linear 参数

In [ ]:
with torch.no_grad():

    logits, _ = model(inputs)

    predictions = logits.argmax(dim=-1)

print("Predictions:")

for i in range(len(sentences)):

    input_words = [
        id_to_word[idx.item()]
        for idx in inputs[i]
    ]

    predicted_words = [
        id_to_word[idx.item()]
        for idx in predictions[i]
    ]

    target_words = [
        id_to_word[idx.item()]
        for idx in targets[i]
    ]

    print("\nInput:     ", input_words)
    print("Prediction:", predicted_words)
    print("Target:    ", target_words)

Word2Vec learns word representations; RNN Language Models learn sequence context for next-token prediction.Word2Vec 和 RNN Language Model 的核心区别是：Word2Vec 主要根据一个词附近的上下文来学习这个词的固定向量表示，重点是“把词表示好”；而 RNN Language Model 会按照词语出现的顺序处理整个序列，用 hidden state 保留前文信息，并根据之前的内容预测下一个 token，重点是“理解上下文并进行序列预测”